# Phase 0: Attention Heatmap 검증

사전학습된 OFT 체크포인트로 LIBERO 프레임 1개를 forward하고, action→vision attention이 조작 대상 물체에 집중하는지 확인한다.

실행 전: 런타임 유형을 GPU로 설정 (런타임 > 런타임 유형 변경 > GPU. 가능하면 A100, 없으면 L4/T4).

In [ ]:
!nvidia-smi

## 1. 설치

In [ ]:
!pip install -q torch torchvision torchaudio

In [ ]:
!git clone https://github.com/airhood/openvla-ivm.git
%cd openvla-ivm
!pip install -q -e .

`torch==2.2.0`은 NumPy 2.0 이전 ABI로 빌드되어 있는데, Colab은 NumPy 2.x가 기본으로 깔려 있어 `Failed to initialize NumPy: _ARRAY_API not found` 경고/오류가 날 수 있다. numpy를 1.x로 고정한다.

**아래 셀 실행 후에도 numpy 관련 경고/에러가 계속 뜨면**: 런타임 > 세션 다시 시작 → 이 노트북을 처음(nvidia-smi)부터 다시 순서대로 실행. (numpy가 이미 import된 세션에서는 재설치만으로 안 바뀔 수 있음)

In [ ]:
!pip install -q "numpy<2"

`pyproject.toml`이 transformers를 moojink의 커스텀 fork(bidirectional attention, parallel decoding용)로 자동으로 잡아준다. 별도 설치 불필요.

flash-attn은 설치하지 않는다 — `output_attentions=True`는 flash attention을 쓰지 못하고 eager/sdpa 경로로 강제 전환되므로 Phase 0에서는 의미가 없다.

In [ ]:
!git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
!pip install -q -e LIBERO
!pip install -q -r experiments/robot/libero/libero_requirements.txt

## 2. 실행

체크포인트는 OFT가 공개한 사전학습 체크포인트를 그대로 쓴다 (Phase 0은 "사전학습 모델의 attention이 원래 조작 대상에 집중하는가"를 보는 것이라 우리가 학습시킬 필요가 없다). 처음 실행 시 HF에서 체크포인트(~15GB)를 자동 다운로드한다.

In [ ]:
!python research/phase0_attention_heatmap.py \
  --pretrained_checkpoint moojink/openvla-7b-oft-finetuned-libero-spatial \
  --task_suite_name libero_spatial \
  --task_id 0 \
  --episode_idx 0 \
  --center_crop \
  --output_dir ./phase0_out

## 3. 결과 확인

In [ ]:
from PIL import Image
display(Image.open("./phase0_out/heatmap_mean.png"))

In [ ]:
from PIL import Image
display(Image.open("./phase0_out/heatmap_grid.png"))

## 4. 판단 기준

- `heatmap_mean.png`에서 밝은 영역이 조작 대상 물체에 걸쳐 있으면 **통과**
- `heatmap_grid.png`에서 32개 head 중 하나라도 물체에 집중하면 **통과** (AttentionMLP가 나중에 그 head를 골라내면 됨)
- 전부 배경/구석에 흩어져 있으면 → `docs/MODEL.md` §9 hidden-state fallback 검토

다른 task_suite/task_id로도 돌려보고 싶으면 위 2번 셀의 인자만 바꿔서 재실행하면 된다.

사용 가능한 체크포인트:
- `moojink/openvla-7b-oft-finetuned-libero-spatial`
- `moojink/openvla-7b-oft-finetuned-libero-object`
- `moojink/openvla-7b-oft-finetuned-libero-goal`
- `moojink/openvla-7b-oft-finetuned-libero-10`